# 📊 Notebook 01 — Data Collection & Inspection

**Project:** PropertyIQ — Production-Grade Property Valuation System for Indian Banks

**Purpose:** Load all raw datasets, inspect them thoroughly, document every column,
identify data quality issues, and save a clean inspection report. This notebook is
the **foundation** every downstream notebook depends on — column names, data types,
null patterns, and city-name conventions discovered here feed directly into
Notebook 02 (Preprocessing) and beyond.

**Author:** PropertyIQ Data Science Team

**Date:** 2026-03-09

**Dataset Sources:**
| # | File | Description |
|---|------|-------------|
| 1 | `Bengaluru_House_Data.csv` | Bengaluru residential sale prices |
| 2 | `Pune house data.csv` | Pune residential sale prices |
| 3 | `Delhi house data.csv` | Delhi residential sale prices |
| 4 | `Mumbai House Prices.csv` | Mumbai residential sale prices |
| 5 | `Real Estate Data V21.csv` | Multi-city raw scrape (property listings) |
| 6 | `House_Rent_Dataset.csv` | Multi-city rental listing data |
| 7 | `QINR628BIS.csv` | RBI All-India House Price Index (via FRED) |

**Expected Outputs:**
- Console inspection report for each dataset
- Cross-dataset compatibility matrix
- Data quality summary table
- `outputs/inspection_report.json` — machine-readable inspection findings

## Cell 2 — Imports & Constants

**Why:** Centralise every dependency and configuration value at the top so the
notebook is deterministic and easy to reconfigure. `pathlib.Path` ensures
cross-platform path handling. The `logging` module gives us timestamped,
levelled output that is far more debuggable than bare `print()`.

In [ ]:
import json
import logging
import sys
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd

# ── Logging ──────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger("PropertyIQ.NB01")

# ── Paths ────────────────────────────────────────────────────
PROJECT_ROOT = Path(__file__).resolve().parent.parent if "__file__" in dir() else Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Dataset registry ────────────────────────────────────────
# Maps a friendly name --> filename. Order matters for reporting.
DATASET_REGISTRY = {
    "Bengaluru": "Bengaluru_House_Data.csv",
    "Pune": "Pune house data.csv",
    "Delhi": "Delhi house data.csv",
    "Mumbai": "Mumbai House Prices.csv",
    "RealEstateV21": "Real Estate Data V21.csv",
    "Rental": "House_Rent_Dataset.csv",
    "HPI": "QINR628BIS.csv",
}

# Minimum expected rows — acts as a sanity gate after loading.
EXPECTED_MIN_ROWS = {
    "Bengaluru": 1_000,
    "Pune": 1_000,
    "Delhi": 500,
    "Mumbai": 1_000,
    "RealEstateV21": 1_000,
    "Rental": 1_000,
    "HPI": 10,
}

# Cities we intend to model
CITIES = ["Bengaluru", "Pune", "Delhi", "Mumbai", "Chennai", "Hyderabad", "Kolkata"]

NULL_WARNING_THRESHOLD = 20.0  # percent

logger.info("Constants initialised — DATA_DIR: %s", DATA_DIR)
assert DATA_DIR.exists(), f"Data directory not found: {DATA_DIR}"

## Cell 3 — Helper Functions

**Why:** Three reusable functions keep every later cell short and consistent.
`load_dataset` handles both CSV and Excel with graceful error handling.
`inspect_dataset` produces a standardised inspection dictionary.
`print_inspection` renders that dictionary in a clean, table-formatted report.

In [ ]:
def load_dataset(filepath: Path, name: str) -> pd.DataFrame:
    """Load a dataset from disk with extension-aware parsing and error handling.

    Args:
        filepath (Path): Absolute path to the data file (.csv or .xlsx).
        name (str): Human-readable dataset name for log messages.

    Returns:
        pd.DataFrame: The loaded dataframe.

    Raises:
        FileNotFoundError: If the file does not exist.
        ValueError: If the file extension is unsupported.

    Example:
        >>> df = load_dataset(DATA_DIR / "Bengaluru_House_Data.csv", "Bengaluru")
    """
    try:
        if not filepath.exists():
            raise FileNotFoundError(f"File not found: {filepath}")

        ext = filepath.suffix.lower()
        if ext == ".csv":
            df = pd.read_csv(filepath)
        elif ext in (".xlsx", ".xls"):
            df = pd.read_excel(filepath)
        else:
            raise ValueError(f"Unsupported file extension '{ext}' for {name}")

        logger.info(
            "Loaded %-20s | %s rows x %s cols | %s",
            name, f"{len(df):,}", len(df.columns), filepath.name,
        )
        return df

    except Exception as exc:
        logger.error("FAILED to load %s: %s", name, exc)
        raise


def inspect_dataset(df: pd.DataFrame, name: str) -> Dict[str, Any]:
    """Produce a comprehensive inspection dictionary for a dataframe.

    Args:
        df (pd.DataFrame): The dataframe to inspect.
        name (str): Human-readable dataset name.

    Returns:
        Dict[str, Any]: Inspection results including shape, dtypes, nulls,
            numeric stats, sample values, and duplicate count.

    Example:
        >>> info = inspect_dataset(df, "Bengaluru")
        >>> info["shape"]
        (13320, 9)
    """
    n_rows, n_cols = df.shape
    dup_count = int(df.duplicated().sum())

    columns_info = {}
    for col in df.columns:
        null_count = int(df[col].isna().sum())
        null_pct = round(null_count / n_rows * 100, 2) if n_rows > 0 else 0.0
        col_info: Dict[str, Any] = {
            "dtype": str(df[col].dtype),
            "null_count": null_count,
            "null_pct": null_pct,
            "sample_values": df[col].dropna().head(3).tolist(),
        }
        if pd.api.types.is_numeric_dtype(df[col]):
            desc = df[col].describe()
            col_info.update({
                "min": float(desc.get("min", np.nan)),
                "max": float(desc.get("max", np.nan)),
                "mean": round(float(desc.get("mean", np.nan)), 2),
                "median": round(float(df[col].median()), 2),
            })
        columns_info[col] = col_info

    return {
        "name": name,
        "shape": {"rows": n_rows, "cols": n_cols},
        "columns": list(df.columns),
        "columns_info": columns_info,
        "duplicate_rows": dup_count,
        "inspected_at": datetime.now().isoformat(),
    }


def print_inspection(info: Dict[str, Any]) -> None:
    """Pretty-print an inspection dictionary with formatted tables and warnings.

    Args:
        info (Dict[str, Any]): Output of ``inspect_dataset()``.

    Returns:
        None

    Example:
        >>> print_inspection(info)
    """
    name = info["name"]
    rows = info["shape"]["rows"]
    cols = info["shape"]["cols"]

    print(f"\n{'=' * 55}")
    print(f"  {name.upper()} DATASET")
    print(f"{'=' * 55}")
    print(f"  Shape        : {rows:,} rows x {cols} columns")
    print(f"  Duplicates   : {info['duplicate_rows']:,} rows")
    print()
    print("  COLUMNS:")

    # Determine column widths dynamically
    col_names = info["columns"]
    max_name = max(len(c) for c in col_names)
    max_name = max(max_name, 6)  # min header width
    w_name = min(max_name + 2, 30)
    w_dtype, w_nulls, w_pct = 10, 7, 8

    border = f"  +{'-' * (w_name + 2)}+{'-' * (w_dtype + 2)}+{'-' * (w_nulls + 2)}+{'-' * (w_pct + 2)}+"
    header = (
        f"  | {'Column':<{w_name}} | {'Dtype':<{w_dtype}} "
        f"| {'Nulls':>{w_nulls}} | {'Null %':>{w_pct}} |"
    )
    print(border)
    print(header)
    print(border.replace("-", "="))

    warnings = []
    for col in col_names:
        ci = info["columns_info"][col]
        null_pct_str = f"{ci['null_pct']:.2f}%"
        print(
            f"  | {col:<{w_name}} | {ci['dtype']:<{w_dtype}} "
            f"| {ci['null_count']:>{w_nulls},} | {null_pct_str:>{w_pct}} |"
        )
        if ci["null_pct"] > NULL_WARNING_THRESHOLD:
            warnings.append((col, ci["null_pct"]))

    print(border)

    if warnings:
        print()
        for col, pct in warnings:
            print(f"  [!] WARNING: column '{col}' has {pct:.1f}% null values")

    # Numeric summaries
    numeric_cols = [
        c for c in col_names if "min" in info["columns_info"][c]
    ]
    if numeric_cols:
        print(f"\n  NUMERIC SUMMARY:")
        for col in numeric_cols:
            ci = info["columns_info"][col]
            print(
                f"    {col:<{w_name}}  "
                f"min={ci['min']:>12,.2f}  max={ci['max']:>12,.2f}  "
                f"mean={ci['mean']:>12,.2f}  median={ci['median']:>12,.2f}"
            )
    print()


logger.info("Helper functions defined: load_dataset, inspect_dataset, print_inspection")

## Cell 4 — Load & Inspect Bengaluru Dataset

**Why:** Bengaluru is our largest single-city sale-price dataset and the primary
training source. We need to confirm the price column exists, understand the
area representation (`total_sqft` is sometimes a range string like "1200-1500"),
and flag any null patterns that will impact downstream feature engineering.

In [ ]:
datasets: Dict[str, pd.DataFrame] = {}
inspections: Dict[str, Dict[str, Any]] = {}

try:
    df_blr = load_dataset(DATA_DIR / DATASET_REGISTRY["Bengaluru"], "Bengaluru")
    datasets["Bengaluru"] = df_blr

    info_blr = inspect_dataset(df_blr, "Bengaluru")
    inspections["Bengaluru"] = info_blr
    print_inspection(info_blr)

    # ── Assertions ──────────────────────────────────────────
    assert df_blr.shape[0] >= EXPECTED_MIN_ROWS["Bengaluru"], (
        f"Bengaluru has only {df_blr.shape[0]} rows — expected >= {EXPECTED_MIN_ROWS['Bengaluru']}"
    )

    # Discover key columns (case-insensitive search)
    cols_lower = {c.lower(): c for c in df_blr.columns}
    price_col = cols_lower.get("price")
    assert price_col is not None, "No 'price' column found in Bengaluru dataset"

    key_cols = {
        "price": price_col,
        "area": cols_lower.get("total_sqft"),
        "size_bhk": cols_lower.get("size"),
        "location": cols_lower.get("location"),
    }
    logger.info("Bengaluru key columns identified: %s", key_cols)

except Exception as exc:
    logger.error("Bengaluru inspection failed: %s", exc)

## Cell 5 — Load & Inspect Pune Dataset

**Why:** Pune uses the same schema as Bengaluru (from the same data source),
but the location column is named `site_location` instead of `location`.
Confirming this difference now saves debugging time in the preprocessing notebook.

In [ ]:
try:
    df_pune = load_dataset(DATA_DIR / DATASET_REGISTRY["Pune"], "Pune")
    datasets["Pune"] = df_pune

    info_pune = inspect_dataset(df_pune, "Pune")
    inspections["Pune"] = info_pune
    print_inspection(info_pune)

    assert df_pune.shape[0] >= EXPECTED_MIN_ROWS["Pune"], (
        f"Pune has only {df_pune.shape[0]} rows — expected >= {EXPECTED_MIN_ROWS['Pune']}"
    )

    cols_lower = {c.lower(): c for c in df_pune.columns}
    price_col = cols_lower.get("price")
    assert price_col is not None, "No 'price' column found in Pune dataset"

    loc_col = cols_lower.get("site_location", cols_lower.get("location"))
    key_cols_pune = {
        "price": price_col,
        "area": cols_lower.get("total_sqft"),
        "size_bhk": cols_lower.get("size"),
        "location": loc_col,
    }
    logger.info("Pune key columns identified: %s", key_cols_pune)

except Exception as exc:
    logger.error("Pune inspection failed: %s", exc)

## Cell 6 — Load & Inspect Delhi Dataset

**Why:** Delhi uses a completely different schema — columns like `Area`, `BHK`,
`Per_Sqft`, `Transaction`, `Status`. Understanding these differences is critical
because the preprocessing notebook must harmonise all city datasets into a
single unified schema.

In [ ]:
try:
    df_del = load_dataset(DATA_DIR / DATASET_REGISTRY["Delhi"], "Delhi")
    datasets["Delhi"] = df_del

    info_del = inspect_dataset(df_del, "Delhi")
    inspections["Delhi"] = info_del
    print_inspection(info_del)

    assert df_del.shape[0] >= EXPECTED_MIN_ROWS["Delhi"], (
        f"Delhi has only {df_del.shape[0]} rows — expected >= {EXPECTED_MIN_ROWS['Delhi']}"
    )

    cols_lower = {c.lower(): c for c in df_del.columns}
    price_col = cols_lower.get("price")
    assert price_col is not None, "No 'price' column found in Delhi dataset"

    key_cols_del = {
        "price": price_col,
        "area": cols_lower.get("area"),
        "bhk": cols_lower.get("bhk"),
        "location": cols_lower.get("locality"),
        "per_sqft": cols_lower.get("per_sqft"),
    }
    logger.info("Delhi key columns identified: %s", key_cols_del)
    logger.info("Delhi unique Transaction types: %s", df_del.get("Transaction", pd.Series()).unique().tolist())

except Exception as exc:
    logger.error("Delhi inspection failed: %s", exc)

## Cell 7a — Load & Inspect Mumbai Dataset

**Why:** Mumbai is the largest single-city dataset (~76K rows) and represents
the most expensive real estate market in India. Its schema (`bhk`, `type`,
`locality`, `area`, `price`, `price_unit`, `region`) differs from both the
Bengaluru/Pune format and the Delhi format. The `price_unit` column (Lac/Cr)
means raw price values need unit-aware conversion in preprocessing.

In [ ]:
try:
    df_mum = load_dataset(DATA_DIR / DATASET_REGISTRY["Mumbai"], "Mumbai")
    datasets["Mumbai"] = df_mum

    info_mum = inspect_dataset(df_mum, "Mumbai")
    inspections["Mumbai"] = info_mum
    print_inspection(info_mum)

    assert df_mum.shape[0] >= EXPECTED_MIN_ROWS["Mumbai"], (
        f"Mumbai has only {df_mum.shape[0]} rows — expected >= {EXPECTED_MIN_ROWS['Mumbai']}"
    )

    cols_lower = {c.lower(): c for c in df_mum.columns}
    assert cols_lower.get("price") is not None, "No 'price' column in Mumbai dataset"

    # Check price_unit distribution — critical for preprocessing
    if "price_unit" in cols_lower:
        unit_col = cols_lower["price_unit"]
        print(f"\n  PRICE UNIT DISTRIBUTION:")
        for val, cnt in df_mum[unit_col].value_counts().items():
            print(f"    {val:<10} : {cnt:>8,} ({cnt / len(df_mum) * 100:.1f}%)")

    key_cols_mum = {
        "price": cols_lower.get("price"),
        "price_unit": cols_lower.get("price_unit"),
        "area": cols_lower.get("area"),
        "bhk": cols_lower.get("bhk"),
        "location": cols_lower.get("locality"),
        "region": cols_lower.get("region"),
    }
    logger.info("Mumbai key columns identified: %s", key_cols_mum)

except Exception as exc:
    logger.error("Mumbai inspection failed: %s", exc)

## Cell 7b — Load & Inspect Real Estate V21 Dataset

**Why:** This is a multi-city raw scrape. We need to determine whether it has
a city/location column that lets us split by city, or whether location info
is embedded in another field. If it lacks a city column, we must decide whether
the data is usable or should be skipped for city-specific modelling.

In [ ]:
try:
    df_v21 = load_dataset(DATA_DIR / DATASET_REGISTRY["RealEstateV21"], "RealEstateV21")
    datasets["RealEstateV21"] = df_v21

    info_v21 = inspect_dataset(df_v21, "RealEstateV21")
    inspections["RealEstateV21"] = info_v21
    print_inspection(info_v21)

    assert df_v21.shape[0] >= EXPECTED_MIN_ROWS["RealEstateV21"], (
        f"RealEstateV21 has only {df_v21.shape[0]} rows"
    )

    # Check for city-level column
    cols_lower = {c.lower(): c for c in df_v21.columns}
    city_candidates = [c for c in cols_lower if c in ("city", "location", "region", "name")]
    print(f"\n  CITY DETECTION:")
    if city_candidates:
        for cc in city_candidates:
            original = cols_lower[cc]
            uniq = df_v21[original].nunique()
            print(f"    Column '{original}' — {uniq} unique values")
            if uniq <= 30:
                print(f"    Top values: {df_v21[original].value_counts().head(10).to_dict()}")
    else:
        print("    No explicit city/region column found.")
        print("    Location info may be embedded in 'Location' or 'Name' columns.")

    # Check if Location column encodes city
    if "Location" in df_v21.columns:
        sample_locs = df_v21["Location"].dropna().head(10).tolist()
        print(f"\n  SAMPLE LOCATIONS: {sample_locs}")

    logger.info("RealEstateV21 — city column present: %s", bool(city_candidates))

except Exception as exc:
    logger.error("RealEstateV21 inspection failed: %s", exc)

## Cell 8 — Load & Inspect Rental Dataset

**Why:** The rental dataset is our **only source with posting dates**, making it
critical for temporal drift analysis in Notebook 05. The `Posted On` column
defines the time window we can use for train/test splitting by date. We also
need to map its city names to our canonical city list.

In [ ]:
try:
    df_rent = load_dataset(DATA_DIR / DATASET_REGISTRY["Rental"], "Rental")
    datasets["Rental"] = df_rent

    info_rent = inspect_dataset(df_rent, "Rental")
    inspections["Rental"] = info_rent
    print_inspection(info_rent)

    assert df_rent.shape[0] >= EXPECTED_MIN_ROWS["Rental"], (
        f"Rental has only {df_rent.shape[0]} rows"
    )

    # ── Date range analysis ─────────────────────────────────
    date_candidates = [c for c in df_rent.columns if "date" in c.lower() or "posted" in c.lower()]
    if date_candidates:
        date_col = date_candidates[0]
        df_rent[date_col] = pd.to_datetime(df_rent[date_col], errors="coerce")
        min_date = df_rent[date_col].min()
        max_date = df_rent[date_col].max()
        print(f"\n  DATE RANGE ({date_col}):")
        print(f"    Earliest : {min_date}")
        print(f"    Latest   : {max_date}")
        print(f"    Span     : {(max_date - min_date).days} days")
        inspections["Rental"]["date_range"] = {
            "column": date_col,
            "min": str(min_date.date()) if pd.notna(min_date) else None,
            "max": str(max_date.date()) if pd.notna(max_date) else None,
        }
    else:
        print("\n  [!] No date column found in Rental dataset")
        logger.warning("Rental dataset has no date column — drift analysis may be limited")

    # ── City distribution ───────────────────────────────────
    city_col_candidates = [c for c in df_rent.columns if "city" in c.lower()]
    if city_col_candidates:
        city_col = city_col_candidates[0]
        print(f"\n  CITY DISTRIBUTION ({city_col}):")
        for city, cnt in df_rent[city_col].value_counts().items():
            print(f"    {city:<20} : {cnt:>6,} ({cnt / len(df_rent) * 100:.1f}%)")
        inspections["Rental"]["cities"] = df_rent[city_col].value_counts().to_dict()

except Exception as exc:
    logger.error("Rental inspection failed: %s", exc)

## Cell 9 — Load & Inspect RBI FRED House Price Index

**Why:** The HPI is our **macro-economic signal** — a quarterly All-India
residential property price index published by the RBI via FRED. We use it
as an external feature and for macro-trend alignment in Notebook 06
(Explainability & Forecast). We need the date range and value range for
normalisation planning.

In [ ]:
try:
    df_hpi = load_dataset(DATA_DIR / DATASET_REGISTRY["HPI"], "HPI")
    datasets["HPI"] = df_hpi

    info_hpi = inspect_dataset(df_hpi, "HPI")
    inspections["HPI"] = info_hpi
    print_inspection(info_hpi)

    assert df_hpi.shape[0] >= EXPECTED_MIN_ROWS["HPI"], (
        f"HPI has only {df_hpi.shape[0]} rows"
    )

    # Identify date & value columns
    date_col = df_hpi.columns[0]  # typically 'observation_date'
    value_col = df_hpi.columns[1] if len(df_hpi.columns) > 1 else None

    df_hpi[date_col] = pd.to_datetime(df_hpi[date_col], errors="coerce")

    print(f"\n  HPI DETAILS:")
    print(f"    Date column  : {date_col}")
    print(f"    Value column : {value_col}")
    print(f"    Date range   : {df_hpi[date_col].min().date()} --> {df_hpi[date_col].max().date()}")
    if value_col:
        hpi_min = df_hpi[value_col].min()
        hpi_max = df_hpi[value_col].max()
        print(f"    HPI range    : {hpi_min:.2f} --> {hpi_max:.2f}")
        print(f"    Total change : {((hpi_max / hpi_min) - 1) * 100:.1f}%")
        inspections["HPI"]["hpi_range"] = {
            "min": float(hpi_min), "max": float(hpi_max),
            "date_min": str(df_hpi[date_col].min().date()),
            "date_max": str(df_hpi[date_col].max().date()),
        }

except Exception as exc:
    logger.error("HPI inspection failed: %s", exc)

## Cell 10 — Cross-Dataset Compatibility Check

**Why:** City-name inconsistency is a major integration risk. "Bangalore" vs
"Bengaluru" vs "BANGALORE" will cause silent join failures if not caught here.
This compatibility matrix gives us a **single visual reference** for what
data is available per city across all datasets.

In [ ]:
print(f"\n{'=' * 70}")
print(f"  CROSS-DATASET CITY COMPATIBILITY MATRIX")
print(f"{'=' * 70}")

# Build city presence mapping
def get_city_names(df: pd.DataFrame, name: str) -> List[str]:
    """Extract city names from a dataset, handling different schemas.

    Args:
        df (pd.DataFrame): The dataset.
        name (str): Dataset name for context.

    Returns:
        List[str]: List of unique city name strings found.

    Example:
        >>> get_city_names(df_rent, "Rental")
        ['Kolkata', 'Mumbai', 'Bangalore', 'Delhi', 'Chennai', 'Hyderabad']
    """
    city_cols = [c for c in df.columns if c.lower() in ("city", "region")]
    if city_cols:
        return df[city_cols[0]].dropna().unique().tolist()
    return []


# Determine implied city from dataset name for single-city datasets
SINGLE_CITY_MAP = {
    "Bengaluru": ["Bengaluru"],
    "Pune": ["Pune"],
    "Delhi": ["Delhi"],
    "Mumbai": ["Mumbai"],
}

city_presence: Dict[str, Dict[str, bool]] = {}
dataset_names = list(DATASET_REGISTRY.keys())

for ds_name in dataset_names:
    if ds_name not in datasets:
        continue
    if ds_name in SINGLE_CITY_MAP:
        cities_in_ds = SINGLE_CITY_MAP[ds_name]
    else:
        cities_in_ds = get_city_names(datasets[ds_name], ds_name)
    for city in cities_in_ds:
        if city not in city_presence:
            city_presence[city] = {}
        city_presence[city][ds_name] = True

# Print matrix
all_cities_found = sorted(city_presence.keys())
ds_short = {
    "Bengaluru": "BLR", "Pune": "PUNE", "Delhi": "DEL", "Mumbai": "MUM",
    "RealEstateV21": "V21", "Rental": "RENT", "HPI": "HPI",
}

header_cols = [ds_short.get(d, d[:4]) for d in dataset_names]
col_w = 7
city_w = 18

print(f"\n  {'CITY':<{city_w}}", end="")
for h in header_cols:
    print(f"  {h:^{col_w}}", end="")
print()
print(f"  {'-' * city_w}", end="")
for _ in header_cols:
    print(f"  {'-' * col_w}", end="")
print()

for city in all_cities_found:
    print(f"  {city:<{city_w}}", end="")
    for ds_name in dataset_names:
        present = city_presence.get(city, {}).get(ds_name, False)
        marker = "YES" if present else "—"
        print(f"  {marker:^{col_w}}", end="")
    print()

# Note naming discrepancies
print(f"\n  NAMING DISCREPANCIES DETECTED:")
rental_cities = get_city_names(datasets.get("Rental", pd.DataFrame()), "Rental")
if "Bangalore" in rental_cities and "Bengaluru" in all_cities_found:
    print("    • 'Bangalore' (Rental) vs 'Bengaluru' (BLR dataset) — needs mapping")
if "Mumbai" in rental_cities:
    print("    • 'Mumbai' present in Rental data — matches Mumbai dataset")

inspections["_compatibility"] = {
    "cities_found": all_cities_found,
    "city_presence": {c: {k: v for k, v in p.items()} for c, p in city_presence.items()},
}

## Cell 11 — Data Quality Summary

**Why:** Before leaving this notebook, we need a single summary table that
tells Notebook 02 exactly what to expect: how many usable rows per dataset,
what percentage of data we lose to nulls, and whether each dataset is worth
processing or should be skipped. This is the **decision gate** for preprocessing.

In [ ]:
print(f"\n{'=' * 80}")
print(f"  DATA QUALITY SUMMARY — ALL DATASETS")
print(f"{'=' * 80}")

CRITICAL_COLS_MAP = {
    "Bengaluru": ["price", "total_sqft"],
    "Pune": ["price", "total_sqft"],
    "Delhi": ["Price", "Area"],
    "Mumbai": ["price", "area"],
    "RealEstateV21": ["Price", "Total_Area"],
    "Rental": ["Rent", "Size", "City"],
    "HPI": ["QINR628BIS"],
}

summary_rows = []
for ds_name in dataset_names:
    if ds_name not in datasets:
        continue
    df = datasets[ds_name]
    total = len(df)

    # Find critical columns (case-insensitive matching)
    cols_lower_map = {c.lower(): c for c in df.columns}
    critical_raw = CRITICAL_COLS_MAP.get(ds_name, [])
    critical_actual = [cols_lower_map.get(c.lower(), c) for c in critical_raw]
    critical_existing = [c for c in critical_actual if c in df.columns]

    if critical_existing:
        usable = int(df[critical_existing].dropna(how="any").shape[0])
    else:
        usable = total

    loss_pct = round((total - usable) / total * 100, 1) if total > 0 else 0.0

    if loss_pct > 50:
        recommendation = "NEEDS CLEANING"
    elif loss_pct > 80:
        recommendation = "SKIP"
    else:
        recommendation = "USABLE"

    summary_rows.append({
        "dataset": ds_name,
        "total_rows": total,
        "usable_rows": usable,
        "loss_pct": loss_pct,
        "critical_cols": critical_existing,
        "recommendation": recommendation,
    })

# Print formatted table
w = {"name": 18, "total": 10, "usable": 10, "loss": 8, "rec": 16}
header = (
    f"  {'Dataset':<{w['name']}} {'Total':>{w['total']}} "
    f"{'Usable':>{w['usable']}} {'Loss %':>{w['loss']}} {'Recommendation':<{w['rec']}}"
)
sep = f"  {'-' * (sum(w.values()) + 4)}"

print(f"\n{sep}")
print(header)
print(sep)
for row in summary_rows:
    print(
        f"  {row['dataset']:<{w['name']}} {row['total_rows']:>{w['total']},} "
        f"{row['usable_rows']:>{w['usable']},} {row['loss_pct']:>{w['loss']}.1f}% "
        f"{row['recommendation']:<{w['rec']}}"
    )
print(sep)

print(f"\n  KEY COLUMNS IDENTIFIED PER DATASET:")
for row in summary_rows:
    print(f"    {row['dataset']:<18} : {row['critical_cols']}")

inspections["_quality_summary"] = summary_rows

## Cell 12 — Save Inspection Report

**Why:** Persisting all findings as a structured JSON file means downstream
notebooks can programmatically reference column names, null percentages, and
dataset recommendations without re-running this entire inspection.

In [ ]:
# Prepare the report — ensure all values are JSON-serialisable
def make_serialisable(obj: Any) -> Any:
    """Recursively convert numpy/pandas types to native Python for JSON.

    Args:
        obj (Any): Object to convert.

    Returns:
        Any: JSON-serialisable version.

    Example:
        >>> make_serialisable(np.int64(42))
        42
    """
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, dict):
        return {k: make_serialisable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [make_serialisable(i) for i in obj]
    if isinstance(obj, pd.Timestamp):
        return str(obj)
    return obj


report = {
    "generated_at": datetime.now().isoformat(),
    "project": "PropertyIQ",
    "notebook": "01_data_collection",
    "datasets": make_serialisable(inspections),
}

report_path = OUTPUT_DIR / "inspection_report.json"
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False, default=str)

assert report_path.exists(), f"Report was not saved to {report_path}"
file_size_kb = report_path.stat().st_size / 1024

print(f"\n{'=' * 55}")
print(f"  INSPECTION REPORT SAVED")
print(f"{'=' * 55}")
print(f"  Path : {report_path}")
print(f"  Size : {file_size_kb:.1f} KB")
print(f"  Keys : {list(report['datasets'].keys())}")
print()

logger.info("Notebook 01 complete — inspection report saved to %s", report_path)